### Passo 1: Processamento de Dados com spaCy

#### Instalação de Ambiente:

In [ ]:
%pip install pandas spacy scikit-learn matplotlib seaborn wordcloud

In [ ]:
!python -m spacy download en_core_web_sm

#### Importação de Libs

In [ ]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from wordcloud import WordCloud 

nlp = spacy.load("en_core_web_sm")

#### Criação e Consumo dos Datasets

In [ ]:
# Base: https://huggingface.co/datasets/zefang-liu/phishing-email-dataset/viewer?views%5B%5D=train

# Utilização de Dataset Local (arquivo bruto)
df = pd.read_csv('dataset\phishing_emails.csv', encoding='utf-8')  

# Visualização prévia dele
df.head()

#### Pré-Processamento do .CSV "Bruto"

In [ ]:
# Função para limpar e processar o texto dos e-mails usando spaCy:
# - converte para minúsculas
# - remove stopwords, pontuações e tokens não alfabéticos
# - aplica lematização
def preprocess_spacy(text):
    if pd.isnull(text):
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return " ".join(tokens)

# Aplica o pré-processamento aos textos da coluna "Email Text"
df["clean_text"] = df["Email Text"].apply(preprocess_spacy) 
df.head()

In [ ]:
# Remove a coluna "Unnamed: 0"
df.drop(columns=["Unnamed: 0"], inplace=True)

# Substitui os valores da coluna "Email Type"
df["Email Type"] = df["Email Type"].replace({
    "Safe Email": 0,
    "Phishing Email": 1
})

df.head()

In [ ]:
# Renomear a coluna "Email Type" para "Phishing?"
df.rename(columns={"Email Type": "Phishing?"}, inplace=True)

# Remover a coluna original "Email Text"
df.drop(columns=["Email Text"], inplace=True)

# Renomear a coluna "clean_text" para "Email Text"
df.rename(columns={"clean_text": "Email Text"}, inplace=True)

# Reordenar as colunas: primeiro "Email Text", depois "Phishing?"
df = df[["Email Text", "Phishing?"]]

In [ ]:
# Exporta o DataFrame pré-processado para um novo CSV
df.to_csv("dataset/emails_processados.csv", index=False, encoding="utf-8")